# Data Collection & EDA

---

## Goal
Collect raw text from official U.S. immigration sources and produce a clean JSON corpus ready for RAG chunking in Week 2.

## Data Sources
| Source | Format | Coverage |
|--------|--------|----------|
| USCIS Policy Manual | HTML | F-1, OPT, STEM OPT, H-1B |
| USCIS AFM PDFs | PDF | H-1B chapters (not yet in Policy Manual) |
| State Dept Visa Bulletin | HTML | Monthly priority date updates |
| Federal Register | REST API | Recent immigration rule notices |

## Output
`/data/raw_docs.json` — list of document dicts with keys: `source`, `label`, `url`, `text`, `word_count`, `char_count`, `scraped_at`

## 1. Install Dependencies

In [1]:
!pip install requests beautifulsoup4 pdfplumber pandas -q

## 2. Imports and Environment Setup

Uses AWS S3 to persist scraped data across SageMaker sessions.
boto3 is pre-installed in SageMaker — no additional setup needed.


In [2]:
import requests
from bs4 import BeautifulSoup
import pdfplumber
import urllib.request
import pandas as pd
import json
import time
import re
import os
import boto3
from datetime import datetime

# Spoof a browser User-Agent to avoid USCIS bot-blocking
headers = {"User-Agent": "Mozilla/5.0"}

# S3 client — SageMaker execution role already has access
s3 = boto3.client("s3", region_name="us-east-1")
S3_BUCKET = "immigration-navigator-data"


## 3. Scrape USCIS Policy Manual

Covers the core MVP pipeline:
- **F-1** student rules → Volume 2, Part F
- **OPT** authorization → Volume 7, Part A
- **STEM OPT** extension → Volume 7, Part F
- **H-1B** overview → Volume 2, Part H
- **Employment Authorization** (EAD/I-765) → Volume 10, Part B

> ⚠️ H-1B individual chapters return 404 — content has not yet been migrated from the legacy AFM. See Part 4.

In [4]:
USCIS_PAGES = {
    # F-1 Student Visa — status rules, maintenance, violations
    "f1_overview":  "https://www.uscis.gov/policy-manual/volume-2-part-f",
    "f1_chapter1":  "https://www.uscis.gov/policy-manual/volume-2-part-f-chapter-1",
    "f1_chapter2":  "https://www.uscis.gov/policy-manual/volume-2-part-f-chapter-2",
    "f1_chapter3":  "https://www.uscis.gov/policy-manual/volume-2-part-f-chapter-3",
    "f1_chapter4":  "https://www.uscis.gov/policy-manual/volume-2-part-f-chapter-4",
    "f1_chapter5":  "https://www.uscis.gov/policy-manual/volume-2-part-f-chapter-5",
    # OPT — pre-completion, post-completion, application windows
    "opt_overview": "https://www.uscis.gov/policy-manual/volume-7-part-a",
    "opt_chapter1": "https://www.uscis.gov/policy-manual/volume-7-part-a-chapter-1",
    "opt_chapter2": "https://www.uscis.gov/policy-manual/volume-7-part-a-chapter-2",
    "opt_chapter3": "https://www.uscis.gov/policy-manual/volume-7-part-a-chapter-3",
    "opt_chapter4": "https://www.uscis.gov/policy-manual/volume-7-part-a-chapter-4",
    "opt_chapter5": "https://www.uscis.gov/policy-manual/volume-7-part-a-chapter-5",
    # STEM OPT extension — eligibility, employer requirements, deadlines
    "stem_opt_ch1": "https://www.uscis.gov/policy-manual/volume-7-part-f-chapter-1",
    "stem_opt_ch2": "https://www.uscis.gov/policy-manual/volume-7-part-f-chapter-2",
    "stem_opt_ch3": "https://www.uscis.gov/policy-manual/volume-7-part-f-chapter-3",
    "stem_opt_ch4": "https://www.uscis.gov/policy-manual/volume-7-part-f-chapter-4",
    # H-1B overview only — detailed chapters are in AFM PDFs (Cell 4)
    "h1b_overview": "https://www.uscis.gov/policy-manual/volume-2-part-h",
    # Employment Authorization Document (EAD) — I-765 rules
    "emp_auth_ch1": "https://www.uscis.gov/policy-manual/volume-10-part-b-chapter-1",
    "emp_auth_ch2": "https://www.uscis.gov/policy-manual/volume-10-part-b-chapter-2",
}

def scrape_uscis_page(url, label):
    """
    Scrape a single USCIS Policy Manual page and return a document dict.

    USCIS renders content inside 'dialog-off-canvas-main-canvas'.
    Nav/header/footer elements are stripped before text extraction.
    Pages returning fewer than 200 words are flagged as short.

    Args:
        url   (str): Full URL of the USCIS policy page.
        label (str): Short identifier used as document key.

    Returns:
        dict: Document with keys: source, label, url, text,
              scraped_at, char_count, word_count.
        None: If the request fails.
    """
    try:
        r = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(r.content, "html.parser")

        # Try selectors in order of reliability on USCIS.gov
        content = (
            soup.find("div", class_="dialog-off-canvas-main-canvas") or
            soup.find("div", class_="field-items") or
            soup.find("article") or
            soup.find("main")
        )

        if content:
            for tag in content.find_all(["nav", "header", "footer", "script", "style"]):
                tag.decompose()
            text = content.get_text(separator="\n", strip=True)
        else:
            text = soup.get_text(separator="\n", strip=True)

        # Normalize whitespace
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'[ \t]+', ' ', text)

        word_count = len(text.split())
        status = "OK" if word_count > 200 else "WARNING! Short"
        print(f"{status} {label}: {word_count} words")

        return {
            "source":     "USCIS Policy Manual",
            "label":      label,
            "url":        url,
            "text":       text,
            "scraped_at": datetime.now().isoformat(),
            "char_count": len(text),
            "word_count": word_count,
        }
    except Exception as e:
        print(f"ERROR {label}: {e}")
        return None

# Scrape all pages — 1.5s delay to be respectful to the server
uscis_docs = []
for label, url in USCIS_PAGES.items():
    doc = scrape_uscis_page(url, label)
    if doc:
        uscis_docs.append(doc)
    time.sleep(1.5)

print(f"\nTotal USCIS docs  : {len(uscis_docs)}")
print(f"Total words       : {sum(d['word_count'] for d in uscis_docs):,}")

OK f1_overview: 2365 words


OK f1_chapter1: 3060 words


OK f1_chapter2: 3678 words


OK f1_chapter3: 5258 words


OK f1_chapter4: 2920 words


...

OK stem_opt_ch4: 4074 words


OK h1b_overview: 1111 words


OK emp_auth_ch1: 1218 words


OK emp_auth_ch2: 1921 words



Total USCIS docs  : 19
Total words       : 69,710


## 4. H-1B Content via AFM PDFs

As of 2024, USCIS has **not finished migrating** H-1B chapter content into the Policy Manual. The detailed guidance lives in the legacy **Adjudicator's Field Manual (AFM)**, published as public PDFs.

- `afm31` → H-1B petition requirements (~27K words)
- `afm34` → Other employment-authorized nonimmigrants (~4K words)

In [6]:
AFM_PDFS = {
    "h1b_afm_ch31": "https://www.uscis.gov/sites/default/files/document/policy-manual-afm/afm31-external.pdf",
    "h1b_afm_ch34": "https://www.uscis.gov/sites/default/files/document/policy-manual-afm/afm34-external.pdf",
}

h1b_afm_docs = []
for label, url in AFM_PDFS.items():
    try:
        # Download PDF to Colab's local /tmp directory
        pdf_path = f"/tmp/{label}.pdf"
        urllib.request.urlretrieve(url, pdf_path)

        # Extract text from all pages
        with pdfplumber.open(pdf_path) as pdf:
            text = "\n".join([page.extract_text() or "" for page in pdf.pages])

        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'[ \t]+', ' ', text)

        h1b_afm_docs.append({
            "source":     "USCIS AFM",
            "label":      label,
            "url":        url,
            "text":       text,
            "scraped_at": datetime.now().isoformat(),
            "char_count": len(text),
            "word_count": len(text.split()),
        })
        print(f"OK {label}: {len(text.split())} words")

    except Exception as e:
        print(f"ERROR {label}: {e}")

uscis_docs += h1b_afm_docs
print(f"\nTotal USCIS + AFM docs : {len(uscis_docs)}")
print(f"Total words            : {sum(d['word_count'] for d in uscis_docs):,}")

OK h1b_afm_ch31: 27077 words


OK h1b_afm_ch34: 4439 words

Total USCIS + AFM docs : 23
Total words            : 132,742


## 5. State Department Visa Bulletin

The Visa Bulletin is published **monthly** and contains priority date cutoffs for employment-based green cards. Relevant for users asking about the H-1B → green card timeline.

We scrape the **6 most recent bulletins** dynamically from the index page.

In [7]:
def fetch_visa_bulletin(num_months=6):
    """
    Scrape the most recent Visa Bulletin issues from the State Department.

    Discovers bulletin URLs dynamically from the index page
    to avoid hardcoding URLs that change monthly.

    Args:
        num_months (int): Number of recent bulletins to scrape.

    Returns:
        list[dict]: List of document dicts, one per bulletin.
    """
    base_url  = "https://travel.state.gov"
    index_url = f"{base_url}/content/travel/en/legal/visa-law0/visa-bulletin.html"

    r = requests.get(index_url, headers=headers)
    soup = BeautifulSoup(r.content, "html.parser")

    # Discover bulletin URLs from the index page
    bulletin_urls = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "visa-bulletin-for" in href:
            full_url = base_url + href if href.startswith("/") else href
            if full_url not in bulletin_urls:
                bulletin_urls.append(full_url)

    print(f"Bulletins found: {len(bulletin_urls)} — scraping latest {num_months}")

    docs = []
    for url in bulletin_urls[:num_months]:
        try:
            r = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(r.content, "html.parser")

            main = (
                soup.find("div", class_="tsg-rwd-main-copy-body") or
                soup.find("main") or
                soup.find("article")
            )

            text = main.get_text(separator="\n", strip=True) if main else soup.get_text()
            text = re.sub(r'\n{3,}', '\n\n', text)
            text = re.sub(r'[ \t]+', ' ', text)

            label = url.split("visa-bulletin-for-")[-1].replace(".html", "")
            docs.append({
                "source":     "State Dept Visa Bulletin",
                "label":      f"visa_bulletin_{label}",
                "url":        url,
                "text":       text,
                "scraped_at": datetime.now().isoformat(),
                "char_count": len(text),
                "word_count": len(text.split()),
            })
            print(f"OK {label}: {len(text.split())} words")
            time.sleep(1)

        except Exception as e:
            print(f"ERROR {url}: {e}")

    return docs

visa_docs = fetch_visa_bulletin(num_months=6)
print(f"\nTotal Visa Bulletin docs: {len(visa_docs)}")

Bulletins found: 290 — scraping latest 6
OK june-2026: 3480 words


OK may-2026: 3195 words


OK april-2026: 3125 words


OK march-2026: 3184 words


OK february-2026: 3159 words


OK january-2026: 3179 words



Total Visa Bulletin docs: 6


## 6. Federal Register (Abstracts via API)

The Federal Register provides a free REST API. We collect **title + abstract** for recent immigration rule notices.

> **Note:** These are short (~100–200 words each) but provide useful signal about recent policy changes.

In [8]:
def fetch_federal_register(keywords, max_results=20):
    """
    Query the Federal Register API for immigration-related notices.

    Returns title + abstract for each result. Full text is not
    fetched here to keep Week 1 scope manageable.

    Args:
        keywords    (list[str]): Search terms to query.
        max_results (int)      : Max documents per keyword.

    Returns:
        list[dict]: List of document dicts.
    """
    all_results = []

    for keyword in keywords:
        params = {
            "conditions[term]": keyword,
            "per_page":         max_results,
            "order":            "relevance",
            "fields[]": ["title", "abstract", "publication_date",
                         "html_url", "document_number", "type"],
        }
        r = requests.get(
            "https://www.federalregister.gov/api/v1/documents",
            params=params,
            timeout=10
        )
        results = r.json().get("results", [])
        print(f"OK '{keyword}': {len(results)} results")

        for item in results:
            text = f"{item.get('title','')}\n\n{item.get('abstract','') or ''}".strip()
            all_results.append({
                "source":           "Federal Register",
                "label":            "federal_register",
                "url":              item.get("html_url", ""),
                "text":             text,
                "publication_date": item.get("publication_date", ""),
                "scraped_at":       datetime.now().isoformat(),
                "word_count":       len(text.split()),
                "char_count":       len(text),
            })
        time.sleep(0.5)

    print(f"\nTotal Federal Register docs: {len(all_results)}")
    return all_results

# Keywords covering the full F-1 → OPT → H-1B pipeline
keywords = ["OPT practical training", "STEM OPT", "H-1B", "F-1 student"]
fed_docs = fetch_federal_register(keywords)

OK 'OPT practical training': 20 results


OK 'STEM OPT': 20 results


OK 'H-1B': 20 results


OK 'F-1 student': 20 results



Total Federal Register docs: 80


## 7. Combine, Normalize, Save & EDA

Merges all sources into a single list, normalizes field names,
saves to S3, and prints a summary EDA.


In [9]:
all_docs = uscis_docs + visa_docs + fed_docs

# Normalize fields
for d in all_docs:
    if "text" not in d or not d["text"]:
        d["text"] = f"{d.get('title','')}\n\n{d.get('abstract','')}".strip()
    if "word_count" not in d:
        d["word_count"] = len(d["text"].split())
    if "char_count" not in d:
        d["char_count"] = len(d["text"])

# Save to S3
s3.put_object(
    Bucket=S3_BUCKET,
    Key="raw_docs.json",
    Body=json.dumps(all_docs, indent=2).encode("utf-8")
)
print(f"Saved {len(all_docs)} documents to S3\n")

# EDA summary
df = pd.DataFrame([{
    "source":     d["source"],
    "label":      d["label"],
    "word_count": d["word_count"],
    "char_count": d["char_count"],
} for d in all_docs])

print("── Documents and words by source ──")
print(df.groupby("source").agg(
    documents=("label",       "count"),
    total_words=("word_count", "sum"),
    avg_words=("word_count",   "mean"),
).round(0))

print("\n── Top 10 documents by size ──")
print(df[["label", "word_count"]]
      .sort_values("word_count", ascending=False)
      .head(10)
      .to_string(index=False))

print("\n── Overall totals ──")
print(f"Total documents  : {len(all_docs)}")
print(f"Total words      : {df['word_count'].sum():,}")
print(f"Total characters : {df['char_count'].sum():,}")

rag_ready = df[df["word_count"] > 200]
print(f"\n── Ready for RAG chunking (>200 words) ──")
print(f"Documents : {len(rag_ready)}")
print(f"Words     : {rag_ready['word_count'].sum():,}")


Saved 109 documents to S3

── Documents and words by source ──
                          documents  total_words  avg_words
source                                                     
Federal Register                 80         8682      109.0
State Dept Visa Bulletin          6        19322     3220.0
USCIS AFM                         4        63032    15758.0
USCIS Policy Manual              19        69710     3669.0

── Top 10 documents by size ──
       label  word_count
h1b_afm_ch31       27077
h1b_afm_ch31       27077
 f1_chapter5        9162
opt_overview        7801
 f1_chapter3        5258
stem_opt_ch2        4902
h1b_afm_ch34        4439
h1b_afm_ch34        4439
stem_opt_ch4        4074
stem_opt_ch3        3829

── Overall totals ──
Total documents  : 109
Total words      : 160,746
Total characters : 1,013,240

── Ready for RAG chunking (>200 words) ──
Documents : 32
Words     : 152,769
